<img src="https://elementos.entornos.net/clientes/ISPC/ispc.png" width="350" height="200">

#**TECNICATURA SUPERIOR EN CIENCIAS DE DATOS E INTELIGENCIA ARTIFICIAL**

##**"PROCESAMIENTO DE IMÁGENES"**

TERCER AÑO - COHORTE 2024


---

## **DESARROLLO Y AVANCE TÉCNICO**

---

## Docente:
Carlos CHARLETTI

## Estudiantes:


*  Allende Olmedo Nicolás
*  Direni Carlos
*  García Carlos
*  Guaraz Emanuel
*  Moreno Raúl
*  Testa Andrea Paola
*  Villalba Valeria Nieves


Junio de 2026

# **🌩️ CCSN Dataset — Normalización para MobileNetV2**
**Dataset:** Cirrus Cumulus Stratus Nimbus (Zhang et al., 2018)  
**Proyecto:** Predicción climática · Río Tercero, Córdoba  
**Clases:** 11 tipos de nube según la OMM  
**Imágenes:** 2543 JPEG · 256×256 px · color RGB

---
### Bloques del notebook
| Bloque | Contenido |
|--------|-----------|
| 1 | Instalación de dependencias |
| 2 | Descarga del dataset desde Kaggle |
| 3 | Inspección del dataset crudo |
| 7 | Prueba técnica de funcionamiento |


> ⚠️ **Antes de ejecutar:** Ir a `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)`

---
## **📦 BLOQUE 1 — Instalación de dependencias**

In [ ]:
# kaggle  → descarga del dataset
# !pip install -q ultralytics dependencia oficial de YOLOv8
# todo lo demás ya viene en Colab (tensorflow, numpy, matplotlib, PIL)

!pip install -q ultralytics
!pip install -q kaggle
!pip install -q keras-cv
!pip install -q tensorflow

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from PIL import Image
import os, zipfile, json
from collections import Counter
import kagglehub
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, Conv2D, MaxPooling2D
import os, zipfile, json
import cv2  # <--- Agregado para procesamiento digital clásico (OpenCV)
from collections import Counter

# Importar MobileNetV2
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D

print(f'✓ TensorFlow: {tf.__version__}')
print(f'✓ GPU disponible: {tf.config.list_physical_devices("GPU")}')
print('✓ Dependencias listas')

# ⚙️ BLOQUE 1: Configuración del Entorno y Gestión de Dependencias

### 📝 Descripción General
Este bloque inicial se encarga de acondicionar el entorno virtual de **Google Colab**, garantizando la disponibilidad y correcta vinculación de todo el ecosistema de librerías de software necesarias para el procesamiento masivo de datos y el posterior entrenamiento de nuestros modelos de **Deep Learning**.

---

### 🚀 Acciones Principales del Script

1. **📦 Instalación de Dependencias Críticas:** Se realiza la descarga e instalación en modo silencioso (`-q`) del motor oficial de Inteligencia Artificial `ultralytics` (YOLOv8), herramientas de automatización de datos (`kagglehub`) y los frameworks avanzados de Computer Vision: `keras-cv` y `tensorflow`.

2. **🛠️ Carga del Stack Tecnológico (Imports):**
   Se importan los módulos base esenciales para la manipulación matricial de imágenes (`numpy`, `OpenCV`, `PIL`), herramientas de visualización gráfica (`matplotlib`), manipulación de archivos del sistema (`os`, `pathlib`) y la suite para el diseño de arquitectura de Redes Neuronales Convolucionales de **TensorFlow / Keras** (incluyendo capas nativas y la red preentrenada **MobileNetV2** para análisis comparativos).

3. **🔬 Auditoría de Hardware:**
   Realiza un control de calidad técnico en tiempo real al verificar la versión exacta de **TensorFlow** activa y validar de forma autónoma la asignación y disponibilidad de la aceleración por hardware mediante la **GPU (Nvidia T4)** de la instancia, asegurando que el pipeline se ejecute a la máxima velocidad de cómputo disponible.


# **YOLO: DETECCIÓN Y CLASIFICACIÓN DE NUBES Y CIELO DESPEJADO**

### Adaptación de YOLOv8 para detección y clasificación de nubes

En esta sección adaptamos el modelo `YOLOV8Detector` de KerasCV para que su capa de salida esté configurada para las 4 clases de nubes que hemos definido (`CLASS_NAMES_TF`).

Es importante recordar que YOLO es un modelo de **detección de objetos**, lo que significa que no solo clasifica una imagen completa, sino que también localiza objetos dentro de ella mediante **cajas delimitadoras (bounding boxes)**. Para entrenar un modelo YOLO de manera efectiva para detectar y clasificar nubes con estas 4 categorías, idealmente necesitaríamos un dataset de imágenes donde cada nube (o cada instancia de nube si hay varias en una imagen) esté anotada con una caja delimitadora y su clase correspondiente.

El `backbone` (`yolo_v8_m_backbone`) se inicializa con pesos pre-entrenados en el dataset "Clud.V3i.yolov8", lo que puede ser un buen punto de partida. Sin embargo, para un rendimiento óptimo en la detección de nubes, este modelo (o al menos su capa de detección) deberá ser **fine-tuneado** con un dataset más voluminoso de nubes anotado con bounding boxes.

###***NOTA:*** El **Dataset** utilizado tiene un orígen aparentemente Asiático, por lo que los nombres de las nubes detectadas figuran en su idioma natal, a menos que no lo encuentren o no esté etiquetada (la Nube) de esa manera, el modelo optará por una de las clasificaciones aportadas por el equipo.

# **⚙️ BLOQUE 2: Carga y Preparación del Dataset YOLO desde Google Drive**

Realizamos la carga del Dataset YOLO que se encuentra en Google Drive. Procedemos a especificar la ruta exacta al archivo `.zip` en el Drive del equipo.

In [ ]:
# 1 # Montar Google Drive, copiar y descomprimir dataset YOLO

from google.colab import drive
import os, zipfile, shutil

# --- Montar Google Drive ---
drive.mount('/content/drive', force_remount=True)

# --- RUTAS ---
FILE_ID     = '1w3gw8YdjQifTty0y2OBC0fMWbN6dr4gQ'
ZIP_NAME    = 'cloud.v3i.yolov8.zip'
LOCAL_ZIP   = f'/content/{ZIP_NAME}'
EXTRACT_DIR = '/content/dataset'

# --- Descargar el ZIP por ID ---
if not os.path.exists(LOCAL_ZIP):
    print("📦 Descargando ZIP desde Drive por ID...")
    !pip install gdown -q
    !gdown {FILE_ID} --output {LOCAL_ZIP}
    print("✅ Descarga completada.")
else:
    print("✅ ZIP ya existe en /content, omitiendo descarga.")

# --- Descomprimir ---
if not os.path.exists(EXTRACT_DIR) or len(os.listdir(EXTRACT_DIR)) == 0:
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    print(f"📂 Descomprimiendo en {EXTRACT_DIR}...")
    with zipfile.ZipFile(LOCAL_ZIP, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print("✅ Descompresión completada.")
else:
    print("✅ Dataset ya descomprimido, omitiendo.")

RUTA_PROYECTO_DRIVE = EXTRACT_DIR
print(f"📂 Dataset listo en: {RUTA_PROYECTO_DRIVE}")

# Sincronización de variable global para los bloques de OpenCV y YOLOv8
DATASET_DIR = EXTRACT_DIR

# ⚙️ BLOQUE 2: Carga y Preparación del Dataset YOLO desde Google Drive

### 📝 Descripción General
Este bloque automatiza el aprovisionamiento de datos en la instancia local de **Google Colab**. En lugar de realizar una lectura directa o una copia carpeta por carpeta desde el almacenamiento virtual de Google Drive (procesos que suelen ser lentos en entornos de computación distribuidos), el script descarga de forma eficiente el dataset empaquetado en formato comprimido (`.zip`) y lo procesa localmente para maximizar la velocidad de lectura del disco durante el entrenamiento de **YOLOv8**.

---

### 🚀 Acciones Principales del Script

1. **🔗 Enlace y Autenticación de Drive:**
   Establece la conexión de infraestructura mediante `drive.mount`, vinculando el entorno de ejecución con la unidad de almacenamiento en la nube del usuario que ejecuta el notebook.

2. **📥 Descarga Eficiente Automatizada (`gdown`):**
   Utiliza la utilidad de nivel de sistema `gdown` para descargar directamente el archivo comprimido del dataset (`cloud.v3i.yolov8.zip`) empleando su identificador único de archivo (`FILE_ID`). El script incluye una lógica de verificación defensiva (`os.path.exists`) para omitir la descarga si el archivo ya se encuentra en la memoria temporal, optimizando el consumo de ancho de banda.

3. **📂 Extracción y Descompresión en Caché Local:**
   A través del módulo nativo `zipfile`, el software extrae los arreglos de imágenes y archivos de configuración estructurados dentro del directorio local de alta velocidad `/content/dataset`. Al igual que en el paso de descarga, cuenta con un control de redundancia que evita la re-extracción innecesaria si la estructura ya está indexada, dejando el pipeline listo (`DATASET_DIR`) para los módulos de visión artificial.

# **PASO 1: VERIFICACIÓN DE DESCOMPRESIÓN DE ARCHIVO .ZIP CON DATASET:**

Realizamos un Mapeo del subdirectorio de Yolo, es decir, de las carpetas descompresas del Dataset, como por ejemplo: Train, Valid y Test, que son los datos que contienen las imagenes con formato YOLO.

In [ ]:
import os

# Verificar qué subcarpetas generó la descompresión (ej. train, valid, test)
subcarpetas = [f for f in os.listdir(EXTRACT_DIR) if os.path.isdir(os.path.join(EXTRACT_DIR, f))]
print(f"Subcarpetas encontradas en el dataset: {subcarpetas}")

# Definimos la ruta de las imágenes de entrenamiento (ajustar si los nombres varían)
# Comúnmente YOLO usa: dataset/train/images
TRAIN_IMAGES_DIR = os.path.join(EXTRACT_DIR, 'train', 'images')

if os.path.exists(TRAIN_IMAGES_DIR):
    total_imgs = len(os.listdir(TRAIN_IMAGES_DIR))
    print(f"✓ Ruta de imágenes de entrenamiento localizada: {TRAIN_IMAGES_DIR}")
    print(f"✓ Total de imágenes para procesar: {total_imgs}")
else:
    # Si la estructura es directa sin subcarpetas 'images', usamos la raíz
    TRAIN_IMAGES_DIR = os.path.join(EXTRACT_DIR, 'train')
    print(f"⚠️ Estructura simplificada. Ruta asignada: {TRAIN_IMAGES_DIR}")

# 🔍 PASO 1: Verificación de Descompresión y Mapeo del Dataset YOLO

### 📝 Descripción General
Este bloque realiza una auditoría de control de calidad sobre el directorio local de alta velocidad para verificar que el proceso de descompresión del archivo `.zip` se haya ejecutado correctamente. El script escanea el entorno, mapea la estructura de subdirectorios nativa de **YOLOv8** (identificando las particiones esenciales de `train`, `valid` y `test`) y calcula el volumen total de datos disponibles para la fase de entrenamiento.

---

### 🚀 Acciones Principales del Script

1. **🗺️ Mapeo Dinámico de Subcarpetas:**
   A través del módulo `os`, el sistema lista y filtra únicamente los directorios físicos generados en la raíz de la extracción. Esto permite validar visualmente que los conjuntos de datos fragmentados en formato YOLO se encuentren presentes y accesibles para los algoritmos del grupo **VisioNet**.

2. **🧠 Enrutamiento Inteligente y Defensivo:**
   El código implementa una lógica condicional de adaptabilidad jerárquica. Primero busca la ruta estándar y óptima de Ultralytics (`dataset/train/images`). Si el dataset fue exportado con una estructura directa o simplificada, el script lo detecta de forma autónoma y reasigna el puntero directamente a la carpeta raíz de entrenamiento (`dataset/train`), evitando que el cuaderno sufra interrupciones o errores de tipo `FileNotFoundError` en los bloques siguientes.

3. **📊 Conteo de Activos Visuales:**
   Una vez consolidada la ruta definitiva de las imágenes de entrenamiento, el software cuantifica el volumen exacto de archivos (`total_imgs`) que ingresarán al pipeline de procesamiento digital e inferencia artificial, dejando el entorno auditado y listo para la fase de análisis morfológico.

#**PASO 2:  INSPECCIONAR LA ESTRUCTURA INTERNA DEL DATASET DESCOMPRESO:**

Realizamos un pipeline básico de procesamiento digital de imágenes aplicado a imágenes de nubes, utilizando OpenCV y matplotlib.

Este proceso podemos:

*   Explorar visualmente las imágenes de nubes.
*   Preprocesar las imágenes (convertirlas a escala de grises).
*   Elemento de listaMejorar el contraste (con ecualización de histograma) para resaltar texturas y detalles de las nubes.
*   Detectar bordes (con el algoritmo de Canny) para identificar la estructura y contornos de las formaciones nubosas.

Se busca visualizar los pasos de cómo se pueden transformar las imágenes para su posterior análisis o entrenamiento de modelos de visión artificial, especialmente enfocado en las características de las nubes.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random

# 1. Seleccionar una imagen al azar del directorio de entrenamiento
lista_imagenes = [f for f in os.listdir(TRAIN_IMAGES_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
imagen_aleatoria = random.choice(lista_imagenes)
ruta_completa = os.path.join(TRAIN_IMAGES_DIR, imagen_aleatoria)

# 2. Cargar la imagen original (OpenCV lee en BGR, lo convertimos a RGB)
img_bgr = cv2.imread(ruta_completa)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# 3. Preprocesamiento: Conversión a Escala de Grises
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

# 4. Mejora de Contraste: Ecualización de Histograma (Ideal para resaltar nubes)
img_equalized = cv2.equalizeHist(img_gray)

# 5. Segmentación / Detección de Bordes: Algoritmo de Canny
# Filtramos el ruido con un desenfoque Gaussiano antes de detectar bordes
img_blur = cv2.GaussianBlur(img_equalized, (5, 5), 0)
bordes_canny = cv2.Canny(img_blur, threshold1=50, threshold2=150)

# 6. Gráfica de control visual para la entrega de la Evidencia
plt.figure(figsize=(15, 10))

plt.subplot(2, 2, 1)
plt.imshow(img_rgb)
plt.title(f"1. Imagen Original (YOLO v3i)\n{imagen_aleatoria}", fontsize=12, fontweight='bold')
plt.axis('off')

plt.subplot(2, 2, 2)
plt.imshow(img_gray, cmap='gray')
plt.title("2. Escala de Grises", fontsize=12, fontweight='bold')
plt.axis('off')

plt.subplot(2, 2, 3)
plt.imshow(img_equalized, cmap='gray')
plt.title("3. Histograma Ecualizado\n(Realce de texturas de nubes)", fontsize=12, fontweight='bold')
plt.axis('off')

plt.subplot(2, 2, 4)
plt.imshow(bordes_canny, cmap='gray')
plt.title("4. Detección de Bordes (Canny)\n(Segmentación de capas nubosas)", fontsize=12, fontweight='bold')
plt.axis('off')

plt.suptitle("Pipeline de Procesamiento Digital de Imágenes - Grupo VisioNet", fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

# 🎨 PASO 2: Pipeline de Procesamiento Digital de Imágenes (PDI) Avanzado

### 📝 Descripción General
Este bloque constituye el núcleo de la ingeniería de características visuales (*Feature Engineering*) del proyecto. Mediante la librería **OpenCV**, se diseña y ejecuta un pipeline secuencial de procesamiento sobre muestras aleatorias del dataset. El objetivo es aislar el ruido cromático del cielo y maximizar los gradientes morfológicos de las masas de vapor, transformando los datos crudos en matrices optimizadas para el posterior análisis y entrenamiento de modelos de visión artificial.

---

### 🚀 Acciones Principales del Script

1. **🔄 Decodificación Espacial (BGR a RGB):**
   OpenCV carga las imágenes de forma nativa en el espacio de color BGR. El script realiza la transposición matricial inmediata al estándar RGB mediante `cv2.cvtColor` para garantizar una visualización cromática realista y exacta en los reportes de control.

2. **🔘 Reducción Dimensional (Escala de Grises):**
   Transforma la matriz tridimensional (canales de color R, G, B) a una representación bidimensional de intensidades lumínicas. Este paso elimina la información redundante del color del cielo (azul/celeste) y reduce drásticamente el costo computacional de las operaciones matemáticas subsiguientes.

3. **📈 Optimización de Contraste (Ecualización de Histograma):**
   Aplica el algoritmo `cv2.equalizeHist` para redistribuir uniformemente las intensidades de los píxeles grises. En la meteorología por visión artificial, esta técnica es clave: estira el contraste en zonas críticas, permitiendo **resaltar microtexturas, densidades y variaciones de vapor** en nubes que originalmente presentaban una iluminación muy plana o sobreexpuesta.

4. **⚡ Segmentación Estructural (Desenfoque Gaussiano + Canny):**
   * **Suavizado:** Se aplica un filtro Gaussiano kernel `(5, 5)` para disipar el ruido de alta frecuencia del sensor fotográfico.
   * **Detección de Bordes:** El algoritmo de **Canny** calcula las derivadas espaciales de la imagen para aislar discontinuidades abruptas de brillo. Esto permite **mapear y segmentar los contornos y límites geométricos** de las formaciones nubosas, abstrayendo su morfología base para el grupo **VisioNet**.

5. **📊 Matriz de Control Visual (Gridspec):**
   Estructura un cuadro comparativo cuádruple (`plt.subplot`) en tiempo real, sirviendo como herramienta de auditoría visual para la entrega de evidencias técnicas y validando la efectividad de cada etapa del pipeline.

#**PASO 3: INFERENCIA YOLOv8 CON CONTROL DE BORDES e IMÁGENES VACÍAS**

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import random
import yaml

# Asegurar definición del directorio raíz
DATASET_DIR = '/content/dataset'
TRAIN_IMAGES_DIR = os.path.join(DATASET_DIR, 'train', 'images')
TRAIN_LABELS_DIR = os.path.join(DATASET_DIR, 'train', 'labels')
YAML_PATH = os.path.join(DATASET_DIR, 'data.yaml')

if not os.path.exists(TRAIN_IMAGES_DIR):
    raise FileNotFoundError(f"No se encontró el directorio en: {TRAIN_IMAGES_DIR}")

# 1. Cargar nombres oficiales del data.yaml de Roboflow
if os.path.exists(YAML_PATH):
    with open(YAML_PATH, 'r') as f:
        data_config = yaml.safe_load(f)
    NOMBRES_DATASET = data_config.get('names', [])
    print(f"✓ Nombres oficiales cargados: {NOMBRES_DATASET}")
else:
    NOMBRES_DATASET = ["Cirrus", "Cumulus", "Stratus", "Nimbus"]
    print("⚠️ Usando nombres de respaldo.")

# 2. Seleccionar una imagen al azar con su respectivo .txt
lista_imagenes = [f for f in os.listdir(TRAIN_IMAGES_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
random.shuffle(lista_imagenes)

imagen_seleccionada = lista_imagenes[0]
base_name = os.path.splitext(imagen_seleccionada)[0]
archivo_etiqueta = f"{base_name}.txt"
ruta_txt = os.path.join(TRAIN_LABELS_DIR, archivo_etiqueta)

# 3. Cargar imagen y obtener dimensiones
ruta_img = os.path.join(TRAIN_IMAGES_DIR, imagen_seleccionada)
img = cv2.imread(ruta_img)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
alto, ancho, _ = img.shape

# 4. Leer anotaciones controlando si el archivo existe y tiene datos
lineas = []
if os.path.exists(ruta_txt):
    with open(ruta_txt, 'r') as f:
        lineas = f.readlines()

print(f"📄 Procesando archivo: {imagen_seleccionada}")

# COLOR CONFIGURATION (RGB)
color_azul = (0, 102, 255)
color_verde = (46, 204, 113)

# 5. CASO A: Archivo vacío o sin nubes catalogadas (Background Image)
if len(lineas) == 0:
    print("🌤️ Estado: Imagen de fondo detectada (Cielo Limpio / Sin nubes anotadas).")
    # Dibujar un recuadro verde perimetral sutil de advertencia
    cv2.rectangle(img_rgb, (10, 10), (ancho - 10, alto - 10), color=color_verde, thickness=2)
    # Colocar cartel informativo en la zona superior
    cv2.rectangle(img_rgb, (10, 10), (280, 40), color_verde, -1)
    cv2.putText(img_rgb, "ANALISIS: Cielo Limpio / Sin Nubes", (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)

# 6. CASO B: La imagen contiene objetos (Nubes con coordenadas)
else:
    for linea in lineas:
        datos = linea.strip().split()
        if len(datos) < 5:
            continue

        clase_id = int(datos[0])
        x_centro, y_centro = float(datos[1]), float(datos[2])
        w_normalizado, h_normalizado = float(datos[3]), float(datos[4])

        # Desnormalizar coordenadas a píxeles de pantalla
        w_pixel = int(w_normalizado * ancho)
        h_pixel = int(h_normalizado * alto)
        x_min = int((x_centro * ancho) - (w_pixel / 2))
        y_min = int((y_centro * alto) - (h_pixel / 2))
        x_max = x_min + w_pixel
        y_max = y_min + h_pixel

        # Recuperar nombre de la nube
        if isinstance(NOMBRES_DATASET, list) and clase_id < len(NOMBRES_DATASET):
            nombre_nube = NOMBRES_DATASET[clase_id]
        else:
            nombre_nube = f"Clase_{clase_id}"

        texto_etiqueta = f"Nube: {nombre_nube}"

        # 1. Dibujar Bounding Box Principal (Caja Azul)
        cv2.rectangle(img_rgb, (x_min, y_min), (x_max, y_max), color=color_azul, thickness=2)

        # Calcular dimensiones exactas del texto dinámico
        (texto_ancho, texto_alto), _ = cv2.getTextSize(texto_etiqueta, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)

        # ─── CONTROL DE BORDES CRÍTICO ───
        # Si la caja toca el techo del mapa, bajamos el texto para que no se corte
        if y_min - texto_alto - 10 < 0:
            y_fondo_min = y_min
            y_fondo_max = y_min + texto_alto + 10
            y_texto_pos = y_min + texto_alto + 5
            print(f"⚠️ Ajuste de borde aplicado para: {nombre_nube} (Caja pegada al límite superior).")
        else:
            y_fondo_min = y_min - texto_alto - 10
            y_fondo_max = y_min
            y_texto_pos = y_min - 5

        # 2. Dibujar fondo del texto adaptado al control de bordes
        cv2.rectangle(img_rgb, (x_min, y_fondo_min), (x_min + texto_ancho + 10, y_fondo_max), color_azul, -1)

        # 3. Estampar texto
        cv2.putText(img_rgb, texto_etiqueta, (x_min + 5, y_texto_pos),
                    fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=0.45,
                    color=(255, 255, 255), thickness=1, lineType=cv2.LINE_AA)

# 7. Graficar e inspeccionar el resultado en el Notebook
plt.figure(figsize=(10, 8))
plt.imshow(img_rgb)
plt.title("Control de Inferencia y Calidad de Datos — VisioNet", fontsize=14, fontweight='bold')
plt.axis('off')
plt.show()

# 🎯 PASO 3: Validación de Ground Truth con Control de Desbordamiento e Imágenes de Fondo

### 📝 Descripción General
Este bloque implementa un motor de renderizado y auditoría visual de etiquetas en formato **YOLO (Ground Truth)**. El script está diseñado bajo un enfoque defensivo para realizar dos tareas críticas en el control de calidad de los datos de **VisioNet**: la identificación automática de imágenes sin objetos (*Background Images*) para el entrenamiento de falsos positivos, y el cálculo geométrico preciso para evitar el desbordamiento de texto en los límites superiores de la imagen (*Bounding Box Overflow*).

---

### 🚀 Acciones Principales del Script

1. **🏷️ Sincronización Dinámica de Metadatos (YAML):**
   Carga el archivo de configuración `data.yaml` para mapear los índices numéricos de las clases a sus nombres meteorológicos oficiales. Cuenta con un arreglo de respaldo (*fallback*) predefinido para asegurar la continuidad del flujo informático ante fallos de lectura.

2. **⚖️ Evaluación Condicional de Escenario (Árbol de Decisión):**
   * **Caso A (Cielo Limpio):** Detecta archivos de anotación de longitud cero. El sistema interpreta la ausencia de vectores como una imagen de fondo (esencial para enseñarle al modelo a no confundir el cielo azul con nubes) y genera un recuadro verde perimetral de advertencia.
   * **Caso B (Nubes Catalogadas):** Procesa de forma secuencial las líneas del archivo `.txt`. Desnormaliza las coordenadas relativas de YOLO ($x, y, w, h$) multiplicándolas por las dimensiones de píxeles reales de la imagen para reconstruir las cajas delimitadoras absolutas ($x_{min}, y_{min}, x_{max}, y_{max}$).

3. **🛠️ Algoritmo de Control de Bordes Crítico:**
   Calcula mediante `cv2.getTextSize` el tamaño del texto dinámico de la etiqueta. Si la nube detectada se encuentra al ras del borde superior ($y_{min} < \text{umbral}$), el script invierte dinámicamente la posición del banner azul y el texto hacia el interior de la caja delimitadora, evitando que la información técnica se corte o quede fuera de la pantalla.

#**PASO 3-1:**

In [ ]:
# ==============================================================================
# PASO 3.1: ENTRENAMIENTO ACELERADO POR HARDWARE (GPU T4 ACTIVADA)
# ==============================================================================
from ultralytics import YOLO
import os

# --- CONTROL DE INFRAESTRUCTURA (Aseguramos que no se oculte la GPU) ---
if 'CUDA_VISIBLE_DEVICES' in os.environ:
    del os.environ['CUDA_VISIBLE_DEVICES']

# 1. Inicializamos YOLOv8 nano para transferencia de aprendizaje
model_ia = YOLO('yolov8n.pt')

# 2. Lanzamos el entrenamiento con tus datos mapeados en data.yaml
YAML_PATH = '/content/dataset/data.yaml'

if os.path.exists(YAML_PATH):
    print("🚀 Iniciando entrenamiento de alta velocidad con el Dataset de VisioNet...")
    print("💡 Nota: Si Colab está bien configurado, verás que inicializa CUDA [device=0].")

    model_ia.train(
        data=YAML_PATH,
        epochs=30,
        imgsz=640,
        batch=16,
        device=0,       # 🔥 CORREGIDO: Fuerza de manera oficial el uso de la GPU Nvidia T4 (Dispositivo 0)
        augment=True    # Activa el aumento de datos para evitar el overfitting
    )
    print("✅ ¡Entrenamiento completado con éxito! Los pesos se guardaron en la ruta correcta.")
else:
    print(f"❌ No se pudo iniciar. No se encuentra el archivo de configuración en {YAML_PATH}")

# 🚀 PASO 3.1: Lanzamiento del Entrenamiento de la Red Neuronal (YOLOv8)

### 📝 Descripción General
Este bloque ejecuta la fase medular del proyecto: la transferencia de aprendizaje (*Transfer Learning*) sobre una arquitectura de red neuronal convolucional de vanguardia. Utilizando el modelo preentrenado **YOLOv8 Nano** de Ultralytics (optimizado para entornos perimetrales y de tiempo real), el sistema ajusta sus pesos sinápticos para especializarse en la detección y localización morfológica de los géneros nubosos definidos por el grupo **VisioNet**.

---

### 🚀 Acciones Principales del Script

1. **🧠 Instanciación del Modelo Base:**
   Carga la topología de red neuronal `yolov8n.pt`. Al utilizar transferencia de aprendizaje, el modelo aprovecha el conocimiento previo de extracción de características abstractas (formas, texturas, bordes geométricos) y lo enfoca en el nuevo dominio meteorológico.

2. **🏋️‍♂️ Parametrización del Pipeline de Entrenamiento (`model.train`):**
   * **Hiperparámetros de Control:** Setea la duración en **30 épocas** (ciclos completos de aprendizaje) y un tamaño de lote (*batch*) de **16 imágenes** simultáneas, equilibrando la convergencia del modelo con la capacidad de memoria del hardware.
   * **Resolución Estándar:** Procesa los vectores espaciales a una dimensión nativa de **640x640 píxeles** (`imgsz=640`).
   * **Aumento de Datos (`augment=True`):** Activa de forma dinámica transformaciones aleatorias en las imágenes (rotaciones, cambios de brillo, espejados) durante el entrenamiento. Esta técnica artificial expande el dataset y previene el sobreajuste (*overfitting*), garantizando que la IA aprenda patrones generales y no memorice las fotos.

3. **🔬 Automatización de Infraestructura:**
   El script valida la existencia del archivo de metadatos `data.yaml` y asienta el pipeline sobre la infraestructura de hardware configurada, compilando y resguardando los pesos matemáticos resultantes (`best.pt` y `last.pt`) de forma autónoma.

#**PASO 3-2:**

In [ ]:
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO
import os
from google.colab import files  # Interfaz nativa para interactuar con la PC local

# Ensure ultralytics is installed
!pip install -q ultralytics

# ==============================================================================
# 1. CARGAR EL MODELO YOLOv8
# ==============================================================================
try:
    ruta_pesos = '/content/runs/detect/train/weights/best.pt'
    if os.path.exists(ruta_pesos):
        model = YOLO(ruta_pesos)
        print("✓ Pesos de VisioNet ('best.pt') cargados con éxito.")
    else:
        model = YOLO('yolov8n.pt')
        print("⚠️ Advertencia: No se encontraron los pesos entrenados. Usando modelo base de respaldo.")
except Exception as e:
    model = YOLO('yolov8n.pt')

# ==============================================================================
# 2. INTERFAZ DE CARGA
# ==============================================================================
print("☁️ BIENVENIDO AL MÓDULO DE INFERENCIA EN VIVO — VISIONET ☁️")
print("Por favor, haga clic en el botón de abajo para subir una foto de nubes desde su computadora...")
print("-" * 80)

# Abre la ventana interactiva para buscar archivos en la PC local
archivos_subidos = files.upload()

if len(archivos_subidos) == 0:
    print("❌ Operación cancelada. No se seleccionó ningún archivo.")
else:
    # Obtenemos el nombre original del archivo que se subió
    nombre_archivo_original = list(archivos_subidos.keys())[0]

    # Definimos nuestra ruta estándar de trabajo interna
    RUTA_IMAGEN_NUEVA = '/content/nueva_nube.jpg'

    # Renombramos el archivo subido para que encaje perfectamente en nuestro pipeline
    if os.path.exists(RUTA_IMAGEN_NUEVA):
        os.remove(RUTA_IMAGEN_NUEVA)  # Borramos si quedó alguna prueba anterior
    os.rename(nombre_archivo_original, RUTA_IMAGEN_NUEVA)

    print("-" * 80)
    print(f"📥 Archivo '{nombre_archivo_original}' recibido y procesado correctamente.")
    print(f"🔍 Analizando con el motor de IA de YOLOv8...")

    # Executar predicción con el motor de IA (umbral permisivo de 5%)
    resultados = model.predict(source=RUTA_IMAGEN_NUEVA, conf=0.05, save=False, verbose=False)

    img_bgr = cv2.imread(RUTA_IMAGEN_NUEVA)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    alto, ancho, _ = img_rgb.shape

    boxes = resultados[0].boxes
    color_azul = (0, 102, 255)
    color_verde = (46, 204, 113)

    # ==============================================================================
    # 4. PROCESAR RESULTADOS CON ESTRATEGIA ADAPTATIVA Y DINÁMICA
    # ==============================================================================
    if len(boxes) == 0:
        # Caso defensivo: Cielo limpio
        print("🌤️ Estado IA: No se detectaron estructuras nubosas significativas.")

        # El marco verde y el texto superior también se adaptan sutilmente al tamaño
        grosor_marco = max(2, int(ancho / 300))
        cv2.rectangle(img_rgb, (10, 10), (ancho - 10, alto - 10), color=color_verde, thickness=grosor_marco)

        escala_texto_alerta = max(0.4, ancho / 1500.0)
        cv2.putText(img_rgb, "REAL-TIME IA: Cielo Limpio / Sin Nubes", (20, int(40 * escala_texto_alerta)),
                    cv2.FONT_HERSHEY_SIMPLEX, escala_texto_alerta, (46, 204, 113), 2, cv2.LINE_AA)
    else:
        print(f"🎯 Se detectaron {len(boxes)} estructuras de nubes de manera autónoma.")

        # Ordenamos las cajas detectadas de izquierda a derecha
        lista_cajas = sorted(boxes, key=lambda b: int(b.xyxy[0][0]))

        for i, box in enumerate(lista_cajas):
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            confianza = box.conf[0].item()
            clase_id = int(box.cls[0].item())

            nombre_clase = model.names[clase_id]
            texto_etiqueta = f"{nombre_clase} ({confianza*100:.1f}%)"

            # ─── AJUSTE DEFENSIVO DE TECHO ───
            if y1 < 5:
                y1 = int(alto * 0.03) if alto * 0.03 > 25 else 25  # Despegue proporcional al alto

            # 1. Dibujar la caja delimitadora (Espesor dinámico según resolución)
            grosor_caja = max(2, int(ancho / 500))
            cv2.rectangle(img_rgb, (x1, y1), (x2, y2), color=color_azul, thickness=grosor_caja)

            # ─── MEJORA DE FUENTE DINÁMICA SEGÚN RESOLUCIÓN ───
            escala_dinamica = max(0.4, ancho / 1600.0)
            espesor_dinamico = max(1, int(ancho / 1200))

            # Calcular dimensiones exactas del texto estirado
            (texto_ancho, texto_alto), _ = cv2.getTextSize(texto_etiqueta, cv2.FONT_HERSHEY_SIMPLEX, escala_dinamica, espesor_dinamico)

            # Control dinámico de superposiciones por esquinas alternadas
            limite_techo_critico = texto_alto + 20
            if i % 2 != 0 and y1 < limite_techo_critico:
                # Esquina inferior izquierda (x1, y2)
                y_fondo_min = y2 - texto_alto - 10
                y_fondo_max = y2
                y_texto_pos = y2 - 5
            else:
                # Esquina superior izquierda (hacia adentro del recuadro)
                y_fondo_min = y1
                y_fondo_max = y1 + texto_alto + 10
                y_texto_pos = y1 + texto_alto + 5

            # 2. Dibujar fondo del cartel adaptado
            cv2.rectangle(img_rgb, (x1, y_fondo_min), (x1 + texto_ancho + 10, y_fondo_max), color_azul, -1)

            # 3. Estampar el texto con la tipografía dinámica calculada
            cv2.putText(img_rgb, texto_etiqueta, (x1 + 5, y_texto_pos),
                        fontFace=cv2.FONT_HERSHEY_SIMPLEX, fontScale=escala_dinamica,
                        color=(255, 255, 255), thickness=espesor_dinamico, lineType=cv2.LINE_AA)

    # ==============================================================================
    # 5. MOSTRAR EL RESULTADO FINAL EN EL LIENZO
    # ==============================================================================
    plt.figure(figsize=(12, 9))
    plt.imshow(img_rgb)
    plt.title("Módulo de Inferencia Adaptativo (YOLOv8) — VisioNet", fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# ☁️ PASO 3.2: Módulo de Inferencia en Vivo e Interfaz Adaptativa (Despliegue Final)

### 📝 Descripción General
Este bloque representa la etapa de explotación y despliegue del modelo en un entorno de producción simulado. Implementa una interfaz interactiva bidireccional que permite a cualquier usuario cargar una imagen inédita desde su almacenamiento local hacia la nube de Google Colab. El motor de IA de **VisioNet** procesa el archivo en tiempo real mediante un pipeline geométrico avanzado, diseñado para corregir distorsiones visuales basándose en la resolución nativa de la imagen introducida.

---

### 🚀 Acciones Principales del Script

1. **💾 Carga de Pesos Óptimos y Resguardo (Fallback):**
   El sistema intenta inicializar el modelo utilizando los coeficientes específicos entrenados (`best.pt`). En caso de no hallar la ruta por inconsistencias en el almacenamiento, activa un mecanismo de seguridad (*try-except*) que levanta la arquitectura base de YOLOv8 para impedir el colapso de la aplicación.

2. **🔌 Interfaz de Carga Nativa (`google.colab.files`):**
   Habilita una pasarela de entrada interactiva para interactuar con el hardware local de la PC del usuario. El script automatiza la captura, el filtrado de nombres y el renombrado estandarizado del archivo (`nueva_nube.jpg`), limpiando registros previos para optimizar la memoria caché.

3. **🧠 Inferencia y Renderizado Adaptativo de Parámetros:**
   El script ejecuta el método `model.predict` con un umbral de confianza altamente sensible (`conf=0.05`). Al recuperar las predicciones, calcula métricas dinámicas vinculadas al ancho y alto de la imagen para automatizar tres variables críticas de diseño:
   * **Espesor Proporcional:** Adapta el grosor de las cajas (*bounding boxes*) y el contorno perimetral de cielo limpio según los píxeles reales de la captura.
   * **Escala de Fuente Dinámica:** Modifica el tamaño físico de la tipografía y de los carteles de fondo para evitar que los textos se vuelvan ilegibles en imágenes de muy alta o muy baja resolución.

4. **🔀 Algoritmo Antisuperposición de Etiquetas:**
   Implementa una lógica matemática de ordenamiento espacial (de izquierda a derecha). Si detecta nubes muy próximas cuyos textos podrían colisionar y volverse indescifrables, utiliza el operador módulo (`i % 2 != 0`) para alternar de forma automática la posición de los carteles informativos (colocando unos en la esquina superior izquierda y otros en la inferior izquierda), garantizando una visualización técnica limpia y profesional.

#**PASO 3-3: RESPALDO DE EJECUCIONES**

In [ ]:
# Guardar los mejores pesos en el Drive para no perder el entrenamiento
import shutil
import os

ruta_origen = '/content/runs/detect/train/weights/best.pt'
carpeta_destino_drive = '/content/drive/MyDrive/CDDEIA - ISPC/2026/1er Cuatrimestre/Procesamiento de IMAGENES_Pesos/'

# Crear la carpeta en tu Drive si no existe
os.makedirs(carpeta_destino_drive, exist_ok=True)

if os.path.exists(ruta_origen):
    shutil.copy(ruta_origen, os.path.join(carpeta_destino_drive, 'best_nubes_yolov8.pt'))
    print("💾 ¡Pesos exportados con éxito a tu Google Drive! Quedaron respaldados como 'best_nubes_yolov8.pt'.")
else:
    print("❌ No se encontraron los pesos para exportar. Verifica si el entrenamiento corrió correctamente.")

# 💾 PASO 3.3: Persistencia y Respaldo de Coeficientes Óptimos (MLOps)

### 📝 Descripción General
Este bloque implementa un mecanismo de persistencia y resguardo de datos (*Data Backup*) en una unidad de almacenamiento permanente. Dado que los entornos virtuales de Google Colab poseen discos temporales y volátiles que se destruyen por completo al finalizar o desconectarse la sesión, este script se encarga de migrar de forma segura los mejores pesos matemáticos obtenidos por la red neuronal hacia el almacenamiento en la nube de Google Drive.

---

### 🚀 Acciones Principales del Script

1. **📂 Verificación y Creación Estructural de Directorios:**
   A través de la librería `os`, el sistema audita la ruta de almacenamiento institucional del equipo (`/content/drive/MyDrive/CDDEIA - ISPC/...`). Si la carpeta de destino no existe en la unidad del usuario, el script la genera de forma dinámica utilizando la función recursiva `os.makedirs`, evitando interrupciones por rutas inexistentes.

2. **⚖️ Validación de Existencia de Activos Corruptos:**
   Aplica una lógica defensiva mediante `os.path.exists` para comprobar que el archivo binario `best.pt` (el cual contiene la matriz de coeficientes y sesgos óptimos calculada por YOLOv8) se encuentre físicamente en el disco local temporal.

3. **🚚 Transferencia Segura de Archivos (`shutil.copy`):**
   Utiliza el módulo de nivel de sistema `shutil` para clonar el archivo de pesos y renombrarlo bajo un estándar claro y descriptivo (`best_nubes_yolov8.pt`). Esto consolida el resguardo del modelo, permitiendo que el grupo **VisioNet** pueda reutilizar los pesos entrenados en futuras fases de inferencia local o despliegue en producción sin necesidad de volver a gastar tiempo y cuota de cómputo en el entrenamiento.

# Detector y Clasificador de Nubes - VisioNet
Aplicación web interactiva en Streamlit para la detección y
clasificación de tipos de nubes utilizando un modelo YOLOv8.

In [ ]:
%%writefile app.py
import streamlit as st
from ultralytics import YOLO
from PIL import Image
import numpy as np
import os

# --- CONFIGURACIÓN DE LA INTERFAZ DE STREAMLIT ---

st.title("🌩️ Detector y Clasificador de Nubes - Grupo VisioNet")
st.write("Sube una imagen del cielo para que el modelo YOLOv8 detecte y clasifique el tipo de nube.")

# --- CONFIGURACIÓN DEL MODELO ---
# Ruta absoluta al archivo de pesos (.pt) almacenado en Google Drive
MODEL_PATH = '/content/drive/MyDrive/CDDEIA - ISPC/2026/1er Cuatrimestre/Procesamiento de IMAGENES_Pesos/best_nubes_yolov8.pt'


@st.cache_resource
def load_model(path):
    """
    Carga el modelo YOLOv8 y lo mantiene en caché de Streamlit
    Anidado para evitar recargas costosas en cada iteración del usuario.
    """
    if os.path.exists(path):
        return YOLO(path)
    else:
        return None


# Intento de carga del modelo de VisioNet
model = load_model(MODEL_PATH)
# --- CONTROL DE FLUJO PRINCIPAL ---

if model is None:
    st.error(f"❌ No se encontró el archivo del modelo en la ruta especificada: {MODEL_PATH}. Asegúrate de haber montado el Drive.")
else:
    st.success("💾 ¡Modelo YOLOv8 cargado con éxito!")

    # 2. Selector de archivos para el usuario
    uploaded_file = st.file_uploader("Elige una imagen...", type=["jpg", "jpeg", "png"])

    # 3. Deslizador para ajustar el umbral de confianza
    confidence_threshold = st.slider(
        "Umbral de Confianza para Detección",
        min_value=0.01, max_value=0.99, value=0.25, step=0.01,
        help="Ajusta este valor para controlar la sensibilidad de la detección. Valores más bajos detectan más objetos, pero con más falsos positivos."
    )

    if uploaded_file is not None:
        # Mostrar imagen original
        image = Image.open(uploaded_file)
        st.image(image, caption='Imagen subida', width=700) # Se cambió use_column_width a width
        st.write("Procesando detección...")

        # Convertir a formato compatible con OpenCV/Ultralytics si es necesario
        img_array = np.array(image)

        # Realizar la predicción con YOLO, usando el umbral de confianza
        results = model(img_array, conf=confidence_threshold)

        # Verificar si hay detecciones
        if len(results[0].boxes) == 0:
            st.warning("😞 No se detectaron nubes con el umbral de confianza actual. Intenta bajar el deslizador.")
        else:
            # YOLOv8 permite renderizar los resultados directamente en una imagen de numpy
            res_plotted = results[0].plot()

            # Mostrar la imagen con las Bounding Boxes (Cajas delimitadoras)
            st.image(res_plotted, caption='Resultados de la Detección', width=700) # Se cambió use_column_width a width

            # Mostrar detalles adicionales en texto si lo deseas
            for box in results[0].boxes:
                class_id = int(box.cls[0])
                label = model.names[class_id]
                confidence = float(box.conf[0])
                st.write(f"✨ Detectado: **{label}** con un **{confidence:.2%}** de confianza.")

In [ ]:
!pip install -q streamlit ultralytics pyngrok

In [ ]:
# ==============================================================================
# CONFIGURACIÓN DEL TÚNEL DE ACCESO PÚBLICO (pyngrok)
# DESCRIPCIÓN: Expone el puerto local de Streamlit a una URL pública externa.
# ==============================================================================

from pyngrok import ngrok

# 1. Autenticación con el servicio de ngrok

# Introduce tu Authtoken de Ngrok aquí
NGROK_TOKEN = "3EpiXCtaXTllFyh4zjTBE7ucg6p_5LuWzJzwoG4dd9LB78HCy"
ngrok.set_auth_token(NGROK_TOKEN)

# Crear el túnel para el puerto por defecto de Streamlit (8501)
public_url = ngrok.connect(8501)
print(f"Haz clic en este enlace para abrir tu aplicación: {public_url}")

# Ejecutar la app de Streamlit
!streamlit run app.py